# ML Optimizer - JHU Module 8 总结和延伸

# 1. Basic Gradient Descent

Training a neural network means minimizing a loss function $L(\theta)$.

The most basic update rule is **gradient descent**:

$$
\theta_{t+1} = \theta_t - \eta \nabla_\theta L(\theta_t)
$$

Where:

- $\theta_t$ : parameters at step $t$
- $\eta$ : learning rate
- $\nabla_\theta L(\theta_t)$ : gradient of the loss

Define:

$$
g_t = \nabla_\theta L(\theta_t)
$$

So the update becomes

$$
\theta_{t+1} = \theta_t - \eta g_t
$$

Problems with plain SGD:

- noisy gradients
- different gradient scales across parameters
- slow convergence in flat directions


# 2. Momentum

Momentum smooths gradients using an exponential moving average.

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t
$$

Update rule:

$$
\theta_{t+1} = \theta_t - \eta m_t
$$

Intuition:

- reduces gradient noise
- accelerates movement along consistent directions
- stabilizes training


# 3. RMSProp

RMSProp adapts the learning rate using gradient magnitude.

$$
v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2
$$

Update rule:

$$
\theta_{t+1} = \theta_t - \eta \frac{g_t}{\sqrt{v_t} + \epsilon}
$$

Effect:

- large gradients → smaller steps
- small gradients → larger steps

This produces **per‑parameter adaptive learning rates**.

# 4. Adam Optimizer

Adam combines **Momentum** and **RMSProp**.

It tracks:

- first moment (mean of gradients)。第一动量 惯性/速度。 梯度指数移动平均值
- second moment (mean of squared gradients) 第二动量，梯度平方的指数移动平均值(uncentered variance)


## Adam Equations

Gradient:

$$
g_t = \nabla_\theta L(\theta_t)
$$

First moment （适应过去的惯性 现在的梯度）:

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t
$$

Second moment （适应过去的路况 现在的路况，default $\beta_2$ 关注长期的观察）:

$$
v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2
$$

Bias correction:

$$
\hat m_t = \frac{m_t}{1-\beta_1^t}
$$

$$
\hat v_t = \frac{v_t}{1-\beta_2^t}
$$

Parameter update:

$$
\theta_{t+1} = \theta_t - \eta \frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
$$


##  Interpretation of Adam Components

| Symbol | Meaning |
|------|------|
| $g_t$ | current gradient |
| $m_t$ | moving average of gradients |
| $v_t$ | moving average of squared gradients |
| $\hat m_t$ | bias‑corrected first moment |
| $\hat v_t$ | bias‑corrected second moment |
| $\epsilon$ | numerical stability constant |

## Adam 优化器参数 $\beta_1$ 与 $\beta_2$ 对比总结

| 参数 | 控制对象 | 默认值 | 潜台词 |
| :--- | :--- | :--- | :--- |
| **$\beta_1$** | **方向 (Direction)** | **0.9** | "我主要听过去的经验（惯性），但也听一点现在的意见。" |
| **$\beta_2$** | **步幅 (Scale)** | **0.999** | "我对路况的判断非常保守，需要很久的观察才能改变我对地形的看法。" |

**一句话概括：**
$\beta_1$ 和 $\beta_2$ 越接近 1，Adam 的“记忆”就越长，走得越稳，但反应越慢；越接近 0，Adam 就越敏感，反应越快，但容易震荡。


## Why Bias Correction Exists

At initialization:

$$
m_0 = 0, \quad v_0 = 0
$$

This causes early estimates to be biased toward zero.

Bias correction compensates for this using

$$
\hat m_t = \frac{m_t}{1-\beta_1^t}
$$

$$
\hat v_t = \frac{v_t}{1-\beta_2^t}
$$

--- 

**简单来说，Bias Correction 是为了解决“冷启动”问题。**

在 Adam 算法中，我们初始化一阶动量 $m_0 = 0$ 和二阶动量 $v_0 = 0$。这会导致在训练初期，估算出的动量值严重**偏向于 0**（偏小）。

### 1. 没有修正会发生什么？
假设真实的梯度一直很大（比如 $g=10$），且 $\beta_1 = 0.9$。
- **$t=0$ (初始化)**: $m_0 = 0$
- **$t=1$**:
  $$ m_1 = 0.9 \cdot m_0 + 0.1 \cdot g_1 = 0.9(0) + 0.1(10) = \mathbf{1} $$

**问题**：真实的梯度是 10，但动量 $m_1$ 只有 1。初始值 0 像一个“黑洞”，把估计值拉向了 0。如果不修正，训练初期的步长会非常小，导致收敛极慢。

### 2. Bias Correction 如何修正？

修正公式通过除以 $(1 - \beta^t)$ 来“放大”这个被低估的值：
$$ \hat{m}_t = \frac{m_t}{1 - \beta_1^t} $$
在 $t=1$ 时：
- 分母：$1 - 0.9^1 = 0.1$
- 修正后：$\hat{m}_1 = 1 / 0.1 = \mathbf{10}$

**结果**：修正后的值 $\mathbf{10}$ 完美还原了真实的梯度大小！
### 3. 长期效果
随着时间 $t$ 增加，$\beta^t$ 趋近于 0，分母 $1 - \beta^t$ 趋近于 1，修正因子自然消失。因此，Bias Correction 仅在训练初期起关键作用。

---

## Adam Advantages

- works well with noisy gradients
- per‑parameter adaptive learning rates
- fast initial convergence
- robust default hyperparameters

## Adam manual demo

**Scenario:**
- Parameter $\theta_0 = 0$
- Learning rate $\eta = 0.1$
- $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$
- Gradients $g_t$:
    - $t=1$: $g_1 = 10$ (large gradient)
    - $t=2$: $g_2 = 10$ (large gradient continues)
---
### Step $t=0$ (Initialization)
- $m_0 = 0$
- $v_0 = 0$
- $\theta_0 = 0$
---
### Step $t=1$
**Input:** $g_1 = 10$
1.  **Update Moments:**
    - $m_1 = \beta_1 m_0 + (1-\beta_1) g_1 = 0.9(0) + 0.1(10) = 1$
    - $v_1 = \beta_2 v_0 + (1-\beta_2) g_1^2 = 0.999(0) + 0.001(100) = 0.1$
2.  **Bias Correction:**
    - $\hat{m}_1 = m_1 / (1 - 0.9^1) = 1 / 0.1 = 10$
    - $\hat{v}_1 = v_1 / (1 - 0.999^1) = 0.1 / 0.001 = 100$
3.  **Update Parameter:**
    - $\theta_1 = \theta_0 - \eta \frac{\hat{m}_1}{\sqrt{\hat{v}_1} + \epsilon} = 0 - 0.1 \frac{10}{\sqrt{100}} = 0 - 0.1(1) = -0.1$

**Observation:** Even though the gradient was 10, the step size was effectively normalized to $\eta$ (0.1) because $\hat{m}/\sqrt{\hat{v}} \approx 1$.

---
### Step $t=2$
**Input:** $g_2 = 10$
1.  **Update Moments:**
    - $m_2 = 0.9(1) + 0.1(10) = 0.9 + 1 = 1.9$
    - $v_2 = 0.999(0.1) + 0.001(100) = 0.0999 + 0.1 = 0.1999$
2.  **Bias Correction:**
    - $\hat{m}_2 = 1.9 / (1 - 0.9^2) = 1.9 / 0.19 = 10$
    - $\hat{v}_2 = 0.1999 / (1 - 0.999^2) \approx 0.1999 / 0.001999 = 100$
3.  **Update Parameter:**
    - $\theta_2 = \theta_1 - 0.1 \frac{10}{\sqrt{100}} = -0.1 - 0.1 = -0.2$

**Observation:** The update remains stable at -0.1. Adam quickly adapts to the scale of the gradient.



# 5. L2 Regularization vs Weight Decay

Loss with L2 regularization:

$$
L_{reg}(\theta) = L(\theta) + \frac{\lambda}{2} ||\theta||^2
$$

Gradient becomes:

$$
\nabla L_{reg} = \nabla L + \lambda \theta
$$

For **SGD**, this is equivalent to weight decay.

But with **Adam**, this regularization term is scaled by the adaptive learning rate mechanism.

This distorts the intended regularization strength.

---
### 为什么说 "scaled by the adaptive learning rate mechanism" 是个问题？

这句话指出了 Adam 优化器在处理 **L2 正则化（Weight Decay）** 时的一个经典问题，这个问题导致了 **AdamW**（Adam with Weight Decay）的诞生。

简单来说，这句话解释了为什么在 Adam 中直接加 L2 正则化（像 SGD 那样）效果不好，甚至会失效。

#### 1. 核心矛盾：谁来控制“惩罚力度”？

*   **SGD 的逻辑（正常逻辑）**：
    *   L2 正则化的目的是：**让权重 $w$ 变小**，防止过拟合。
    *   在 SGD 中，更新公式是：
        $$ w_{new} = w_{old} - \eta \cdot (\nabla L + \lambda w) $$
    *   这里 $\lambda w$ 就是正则化项。它的作用非常直接：每次更新都减去一点点 $w$（乘以学习率 $\eta$）。
    *   **结果**：权重会均匀地、稳定地衰减。

*   **Adam 的逻辑（出问题的地方）**：
    *   Adam 的特点是 **“自适应学习率”**。它会根据梯度的历史大小（二阶矩 $v_t$）来缩放每一步的更新幅度。
    *   如果把 L2 正则化项 $\lambda \theta$ 加到梯度里（像 SGD 那样），Adam 会把它当成梯度的一部分来处理。
    *   假设 $\hat{m}_t \approx g_t$ 以简化理解：
        $$ \theta_{t+1} = \theta_t - \eta \cdot \frac{\nabla L(\theta_t) + \lambda \theta_t}{\sqrt{\hat{v}_t} + \epsilon} $$
    *   展开来看：
        $$ \theta_{t+1} = \theta_t - \underbrace{\eta \cdot \frac{\nabla L(\theta_t)}{\sqrt{\hat{v}_t} + \epsilon}}_{\text{正常的梯度更新}} - \underbrace{\eta \cdot \frac{\lambda \theta_t}{\sqrt{\hat{v}_t} + \epsilon}}_{\text{被扭曲的正则化项}} $$
    *   **问题来了**：正则化项 $\lambda \theta_t$ 本来应该直接减去（像 SGD 那样 $-\eta \lambda \theta_t$），结果现在它也被除以了 $\sqrt{\hat{v}_t}$。
    *   **后果**：
        *   如果某个参数的梯度很大（$v_t$ 很大），正则化项 $\lambda \theta$ 就会被除以一个大数，导致**惩罚力度变小**。
        *   如果某个参数的梯度很小（$v_t$ 很小），正则化项 $\lambda \theta$ 就会被除以一个小数，导致**惩罚力度变大**。

#### 2. 为什么这“不好”？

这句话 "scaled by the adaptive learning rate mechanism" 的坏处在于：

1.  **惩罚力度不可控**：你本来希望对所有参数一视同仁地施加 L2 惩罚（比如都衰减 1%），但 Adam 却根据参数的活跃程度（梯度大小）**擅自修改了惩罚力度**。
2.  **大参数逃脱惩罚**：通常我们希望惩罚那些幅度大的参数。但在 Adam 中，幅度大的参数往往梯度也大（$v_t$ 大），结果正则化项反而被缩小了，导致**该惩罚的没惩罚到**。
3.  **小参数被过度惩罚**：不活跃的参数（梯度小，$v_t$ 小）反而被施加了巨大的正则化压力，可能导致它们直接归零，失去了表达能力。

#### 3. 形象比喻(Gemini 解释)

*   **SGD 的 L2 正则化**：像是**“收税”**。不管你赚多少钱，每个人都按固定比例交税（比如 10%）。这很公平，能有效控制贫富差距（防止过拟合）。
*   **Adam 的 L2 正则化（错误版）**：像是**“根据你的消费能力来收税”**。
    *   如果你赚得多（梯度大），Adam 觉得你消费高，于是**给你减税**（除以大 $v_t$）。
    *   如果你赚得少（梯度小），Adam 觉得你消费低，于是**给你加税**（除以小 $v_t$）。
    *   **结果**：富人（大参数）越来越富，穷人（小参数）直接破产。这完全违背了正则化的初衷！

#### 4. 解决方案：AdamW

为了解决这个问题，**AdamW** 出现了。它的做法是：**把 L2 正则化从梯度更新中剥离出来**。

*   **AdamW 的逻辑**：
    1.  先用 Adam 的自适应机制计算梯度的更新量（不包含正则化）。
    2.  **独立地**、**直接地**对权重 $w$ 进行衰减：
        $$ w_{new} = w_{old} - \eta \cdot (\text{Adam更新量}) - \eta \lambda w $$
    *   **效果**：这样，正则化项 $\lambda w$ 就不会被 $\sqrt{v_t}$ 缩放了，恢复了它原本“公平收税”的作用。

**总结：**
那句话之所以描述了一个“不好”的现象，是因为**自适应学习率机制（除以 $\sqrt{v_t}$）干扰了 L2 正则化的原本意图**，导致正则化失效或产生副作用。这就是为什么现在大家都用 **AdamW** 而不是 Adam 的原因。

---

# 6. AdamW

AdamW fixes the issue by **decoupling weight decay from gradient updates**.

Instead of adding the penalty to the gradient, the parameters are directly shrunk.

## AdamW Update Rule

Gradient step:

$$
\theta_{t+1} = \theta_t - \eta \frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
$$

Weight decay step:

$$
\theta_{t+1} = \theta_{t+1} - \eta \lambda \theta_t
$$

Combined form:

$$
\theta_{t+1} = (1-\eta\lambda)\theta_t - \eta \frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
$$


# 7. Adam vs AdamW

| Feature | Adam | AdamW |
|------|------|------|
| Weight decay method | added to gradient | decoupled |
| Regularization behavior | distorted by adaptive scaling | consistent |
| Modern usage | older codebases | modern standard |


# 8. Typical Hyperparameters

| Parameter | Typical Value |
|------|------|
| learning rate | 1e‑3 (small models) |
| $\beta1$ | 0.9 |
| $\beta2$ | 0.999 |
| ε | 1e‑8 |
| weight decay | 0.01 |


# 9. Summary

**Adam** uses exponential moving averages of gradients and squared gradients to produce adaptive learning rates.

**AdamW** improves Adam by separating weight decay from gradient scaling, producing more reliable regularization.

For most modern deep learning systems (Transformers, LLMs, Vision Transformers), **AdamW is the recommended optimizer**.